# 🚀 Module 4 Lab — LoanBot v1.0 Production Deployment

**FinTech Corp AI Engineering Lab**

---

Trong bài lab này, bạn sẽ đóng vai **DevOps Engineer tại FinTech Corp**, chuẩn bị LoanBot từ prototype (v0.2, Module 3) lên **production deployment (v1.0)**.

## Mục tiêu
1. **Production Readiness Check** — kiểm tra LoanBot v1.0 qua 5-group checklist
2. **Docker Configuration** — tạo Dockerfile hợp lệ với security best practices
3. **CI/CD Pipeline** — thiết kế và simulate Quality Gate
4. **Monitoring Dashboard** — tính toán và visualize metrics thực tế
5. **Scaling Calculator** — tính toán capacity planning và cost optimization

## Setup
```
pip install anthropic matplotlib pandas
```

**LoanBot Context:**
- Xử lý hồ sơ vay vốn cho FinTech Corp
- Tools: check_credit_score, verify_income, calculate_dti, check_blacklist, generate_report
- HITL Gateway: hồ sơ > 500 triệu VNĐ cần human approval
- SLA: P95 latency ≤ 300s, Faithfulness ≥ 99%, F1 ≥ 95%

---
## Part 1: Production Readiness Check

Trước khi deploy bất kỳ AI agent nào, chúng ta phải chạy **Production Readiness Checklist** — một quy trình kiểm tra có hệ thống bao gồm 5 nhóm:
- **Correctness**: F1, Faithfulness, Edge cases
- **Security**: Auth, Input sanitization, Secrets management
- **Performance**: Latency SLA, Load test, Memory
- **Compliance**: Legal review, AI disclosure, Audit logs
- **Reliability**: HITL, Retry, Circuit breaker, Health check

Mỗi item: PASS / FAIL / WARNING. **Compliance items là BLOCKING.**

In [ ]:
from dataclasses import dataclass, field
from typing import List, Dict, Optional
from enum import Enum
import json

class CheckStatus(Enum):
    PASS    = "✅ PASS"
    FAIL    = "❌ FAIL"
    WARNING = "⚠️  WARN"
    SKIP    = "⏭️  SKIP"

@dataclass
class CheckItem:
    group: str
    name: str
    requirement: str
    blocking: bool
    status: CheckStatus = CheckStatus.SKIP
    actual_value: str = ""
    note: str = ""

class ProductionReadinessChecker:
    def __init__(self, agent_name: str, version: str):
        self.agent_name = agent_name
        self.version = version
        self.checks: List[CheckItem] = []
        self._define_checks()
    
    def _define_checks(self):
        """Define all 15 production readiness checks."""
        self.checks = [
            # Group 1: Correctness
            CheckItem("Correctness", "F1 Score",
                      "F1 ≥ 95% trên eval suite 100 test cases", blocking=False),
            CheckItem("Correctness", "Faithfulness Score",
                      "Faithfulness ≥ 99%: không hallucinate số liệu", blocking=True),
            CheckItem("Correctness", "Edge Case Coverage",
                      "Xử lý đúng: blacklist, incomplete data, network errors", blocking=False),
            # Group 2: Security
            CheckItem("Security", "API Key Management",
                      "Keys từ Secrets Manager, không hardcode trong code/Docker", blocking=True),
            CheckItem("Security", "Input Sanitization",
                      "Validation cho tất cả tool inputs, reject injections", blocking=True),
            CheckItem("Security", "Data Encryption",
                      "CCCD, income data encrypted at rest và in transit (TLS 1.3)", blocking=False),
            # Group 3: Performance
            CheckItem("Performance", "Latency SLA",
                      "P95 latency ≤ 300s khi concurrent 10 requests", blocking=False),
            CheckItem("Performance", "Load Test",
                      "Stable tại 50 requests/hour trong 8 giờ liên tục", blocking=False),
            CheckItem("Performance", "Memory Leak Check",
                      "Memory usage < 2GB sau 1000 requests", blocking=False),
            # Group 4: Compliance (BLOCKING)
            CheckItem("Compliance", "Legal Review",
                      "Legal team đã review và sign-off AI deployment", blocking=True),
            CheckItem("Compliance", "AI Disclosure",
                      "Users được thông báo đang tương tác với AI agent", blocking=True),
            CheckItem("Compliance", "Audit Log Completeness",
                      "100% decisions có trace_id và JSON audit trail", blocking=True),
            # Group 5: Reliability
            CheckItem("Reliability", "HITL Gateway Test",
                      "20 hồ sơ >500M: 100% trigger HITL đúng", blocking=True),
            CheckItem("Reliability", "Circuit Breaker Test",
                      "Simulate 3 CIC failures → OPEN state, recovery sau 10s", blocking=False),
            CheckItem("Reliability", "Health Check Endpoint",
                      "/health trả về {status: healthy} trong <1s", blocking=False),
        ]
    
    def record_check(self, name: str, status: CheckStatus, actual: str, note: str = ""):
        for check in self.checks:
            if check.name == name:
                check.status = status
                check.actual_value = actual
                check.note = note
                return
        raise ValueError(f"Check '{name}' not found")
    
    def evaluate(self) -> Dict:
        blocking_fails = [c for c in self.checks if c.blocking and c.status == CheckStatus.FAIL]
        total_pass = sum(1 for c in self.checks if c.status == CheckStatus.PASS)
        total_fail = sum(1 for c in self.checks if c.status == CheckStatus.FAIL)
        total_warn = sum(1 for c in self.checks if c.status == CheckStatus.WARNING)
        can_deploy = len(blocking_fails) == 0
        return {
            "can_deploy": can_deploy,
            "verdict": "🚀 READY TO DEPLOY" if can_deploy else "🛑 BLOCKED — DO NOT DEPLOY",
            "stats": {"pass": total_pass, "fail": total_fail, "warn": total_warn},
            "blocking_fails": [c.name for c in blocking_fails]
        }
    
    def print_report(self):
        print(f"\n{'='*60}")
        print(f"  Production Readiness: {self.agent_name} {self.version}")
        print(f"{'='*60}")
        current_group = ""
        for c in self.checks:
            if c.group != current_group:
                current_group = c.group
                block_tag = " [BLOCKING]" if any(x.blocking for x in self.checks if x.group == c.group) else ""
                print(f"\n── {c.group}{block_tag} ──")
            tag = " [BLOCK]" if c.blocking else ""
            print(f"  {c.status.value:<12} {c.name}{tag}")
            if c.actual_value:
                print(f"              → {c.actual_value}")
            if c.note:
                print(f"              Note: {c.note}")
        result = self.evaluate()
        print(f"\n{'='*60}")
        print(f"  VERDICT: {result['verdict']}")
        stats = result['stats']
        print(f"  Stats: {stats['pass']} PASS | {stats['fail']} FAIL | {stats['warn']} WARN")
        if result['blocking_fails']:
            print(f"  BLOCKING ISSUES: {', '.join(result['blocking_fails'])}")
        print(f"{'='*60}\n")

# --- Simulate LoanBot v1.0 readiness results ---
checker = ProductionReadinessChecker("LoanBot", "v1.0")

checker.record_check("F1 Score", CheckStatus.PASS, "F1 = 95.5%, Precision=96.9%, Recall=94.2%")
checker.record_check("Faithfulness Score", CheckStatus.PASS, "Faithfulness = 99.2% trên 100 eval cases")
checker.record_check("Edge Case Coverage", CheckStatus.PASS, "Blacklist ✓, Incomplete data ✓, Network error ✓")
checker.record_check("API Key Management", CheckStatus.PASS, "ANTHROPIC_API_KEY từ AWS Secrets Manager")
checker.record_check("Input Sanitization", CheckStatus.PASS, "VALIDATION_RULES dict reject invalid inputs")
checker.record_check("Data Encryption", CheckStatus.WARNING, "TLS 1.3 ✓, At-rest encryption đang pending",
                     note="Security team đang config KMS, ETA 1 tuần")
checker.record_check("Latency SLA", CheckStatus.PASS, "P95 = 187s (target ≤ 300s) ✓")
checker.record_check("Load Test", CheckStatus.PASS, "50 req/hour × 8h = stable, no memory leak")
checker.record_check("Memory Leak Check", CheckStatus.PASS, "Peak memory: 1.2GB / 2GB limit")
checker.record_check("Legal Review", CheckStatus.PASS, "Legal team sign-off ngày 2026-06-20")
checker.record_check("AI Disclosure", CheckStatus.PASS, "Disclosure banner trong loan application portal")
checker.record_check("Audit Log Completeness", CheckStatus.PASS, "100% requests có trace_id, stored in ELK")
checker.record_check("HITL Gateway Test", CheckStatus.PASS, "20/20 hồ sơ >500M trigger HITL đúng (100%)")
checker.record_check("Circuit Breaker Test", CheckStatus.PASS, "OPEN sau 3 failures, recovery sau 10s ✓")
checker.record_check("Health Check Endpoint", CheckStatus.PASS, "/health P95 latency = 43ms")

checker.print_report()

---
## Part 2: Docker Configuration Generator

Thay vì viết Dockerfile thủ công, chúng ta sẽ tạo một **DockerfileGenerator** tự động tạo production-grade Dockerfile với:
- Multi-stage build (builder + runtime)
- Non-root user
- HEALTHCHECK endpoint
- Security hardening (no secrets in image)
- docker-compose.yml cho local development

In [ ]:
from dataclasses import dataclass
from typing import List, Dict

@dataclass
class DockerConfig:
    app_name: str
    python_version: str
    port: int
    health_check_path: str
    env_secrets: List[str]  # secrets from external (NOT baked in)
    env_config: Dict[str, str]  # non-secret config (OK in image)
    requirements: List[str]
    main_module: str

class DockerfileGenerator:
    def __init__(self, config: DockerConfig):
        self.config = config
        self._security_issues = []
    
    def generate_dockerfile(self) -> str:
        c = self.config
        secret_env_lines = "\n".join([
            f"# {s} — NOT baked in, loaded from Secrets Manager at runtime"
            for s in c.env_secrets
        ])
        config_env_lines = "\n".join([
            f"ENV {k}={v}" for k, v in c.env_config.items()
        ])
        req_file = "\n".join(c.requirements)
        
        dockerfile = f"""# ========================================
# {c.app_name} — Production Dockerfile
# Multi-stage build: builder + runtime
# Security: non-root, no secrets in image
# ========================================

# === Stage 1: Builder ===
# Purpose: install deps with full build tools
FROM python:{c.python_version}-slim AS builder

WORKDIR /build

# Install build dependencies
RUN apt-get update && apt-get install -y --no-install-recommends \\
    gcc libffi-dev && \\
    rm -rf /var/lib/apt/lists/*

# Copy & install Python dependencies
COPY requirements.txt .
RUN pip install --user --no-cache-dir -r requirements.txt

# === Stage 2: Runtime ===
# Purpose: minimal image, only runtime artifacts
FROM python:{c.python_version}-slim AS runtime

WORKDIR /app

# Create non-root user (security best practice)
RUN addgroup --system loanbot && adduser --system --ingroup loanbot loanbot

# Copy only installed packages from builder (not build tools)
COPY --from=builder /root/.local /home/loanbot/.local

# Copy application source
COPY --chown=loanbot:loanbot . .

# Non-secret config (safe to embed in image)
{config_env_lines}

# === SECRETS — DO NOT EMBED ===
{secret_env_lines}
# Loaded at runtime via:
#   docker run --env-file .env ...  (dev)
#   AWS Secrets Manager / Vault     (production)

# Health check (allows orchestrators to monitor container health)
HEALTHCHECK --interval=30s --timeout=10s --start-period=60s --retries=3 \\
    CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:{c.port}{c.health_check_path}')" \\
    || exit 1

# Switch to non-root user
USER loanbot

EXPOSE {c.port}

# Start application
CMD ["python", "-m", "{c.main_module}"]
"""
        return dockerfile
    
    def generate_requirements_txt(self) -> str:
        return "\n".join(self.config.requirements) + "\n"
    
    def generate_docker_compose(self) -> str:
        c = self.config
        secret_env = "\n      ".join([f"- {s}" for s in c.env_secrets])
        return f"""version: '3.9'

services:
  {c.app_name}-api:
    build:
      context: .
      target: runtime
    ports:
      - "{c.port}:{c.port}"
    environment:
      {secret_env}
    env_file:
      - .env.local  # gitignored, dev only
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:{c.port}{c.health_check_path}"]
      interval: 30s
      timeout: 10s
      retries: 3
    restart: unless-stopped
    depends_on:
      redis:
        condition: service_healthy

  redis:
    image: redis:7-alpine
    command: redis-server --requirepass ${{REDIS_PASSWORD}}
    healthcheck:
      test: ["CMD", "redis-cli", "ping"]
      interval: 10s
      timeout: 5s
      retries: 3

  prometheus:
    image: prom/prometheus:latest
    ports:
      - "9090:9090"
    volumes:
      - ./monitoring/prometheus.yml:/etc/prometheus/prometheus.yml
"""
    
    def security_scan(self) -> List[str]:
        issues = []
        dockerfile = self.generate_dockerfile()
        for secret in self.config.env_secrets:
            if f"ENV {secret}=" in dockerfile:
                issues.append(f"CRITICAL: Secret '{secret}' hardcoded in ENV")
        if "USER root" in dockerfile or "USER 0" in dockerfile:
            issues.append("HIGH: Running as root user")
        if "HEALTHCHECK" not in dockerfile:
            issues.append("MEDIUM: No HEALTHCHECK defined")
        return issues if issues else ["✅ No security issues found"]

# Configure LoanBot v1.0
loanbot_docker = DockerfileGenerator(DockerConfig(
    app_name="loanbot",
    python_version="3.11",
    port=8080,
    health_check_path="/health",
    env_secrets=["ANTHROPIC_API_KEY", "CIC_API_KEY", "DATABASE_PASSWORD", "REDIS_PASSWORD"],
    env_config={
        "APP_ENV": "production",
        "LOG_LEVEL": "INFO",
        "MAX_CONCURRENT_REQUESTS": "10",
        "HITL_THRESHOLD_VND": "500000000",
        "MAX_RETRIES": "3",
    },
    requirements=[
        "anthropic==0.40.0",
        "fastapi==0.115.0",
        "uvicorn[standard]==0.32.0",
        "redis==5.2.0",
        "prometheus-client==0.21.0",
        "pydantic==2.10.0",
        "httpx==0.28.0",
    ],
    main_module="loanbot.api"
))

print("=== DOCKERFILE ===\n")
print(loanbot_docker.generate_dockerfile())

print("=== SECURITY SCAN ===\n")
for issue in loanbot_docker.security_scan():
    print(f"  {issue}")
print()

In [ ]:
print("=== DOCKER COMPOSE ===\n")
print(loanbot_docker.generate_docker_compose())

print("=== REQUIREMENTS.TXT ===\n")
print(loanbot_docker.generate_requirements_txt())

---
## Part 3: CI/CD Pipeline Simulator

Simulate 7-stage CI/CD pipeline cho LoanBot v1.0:
1. Code Push
2. Lint & Type Check
3. Docker Build
4. **Eval Suite (Quality Gate)**
5. Staging Deploy
6. Canary Deploy (10%)
7. Full Production Rollout

In [ ]:
import time
import random
from dataclasses import dataclass
from typing import Callable, Optional

@dataclass
class PipelineStage:
    name: str
    duration_s: float
    check_fn: Optional[Callable] = None
    is_quality_gate: bool = False

class CICDPipelineSimulator:
    def __init__(self, app: str, version: str, branch: str):
        self.app = app
        self.version = version
        self.branch = branch
        self.stages = []
        self.results = []
        self.passed = True
    
    def add_stage(self, stage: PipelineStage):
        self.stages.append(stage)
    
    def run(self, eval_metrics: dict) -> bool:
        print(f"\n{'='*65}")
        print(f"  🚀 CI/CD Pipeline: {self.app} {self.version}")
        print(f"  Branch: {self.branch} | {len(self.stages)} stages")
        print(f"{'='*65}")
        
        for i, stage in enumerate(self.stages, 1):
            gate_tag = " [QUALITY GATE]" if stage.is_quality_gate else ""
            print(f"\n[{i}/{len(self.stages)}] {stage.name}{gate_tag}")
            print(f"  ⏳ Running... ({stage.duration_s}s)")
            
            if not self.passed:
                print(f"  ⏭️  SKIPPED (pipeline blocked)")
                self.results.append({"stage": stage.name, "status": "SKIPPED"})
                continue
            
            # Run the check function if provided
            passed_stage = True
            message = "OK"
            if stage.check_fn:
                passed_stage, message = stage.check_fn(eval_metrics)
            
            status = "PASS" if passed_stage else "FAIL"
            icon = "✅" if passed_stage else "❌"
            print(f"  {icon} {status}: {message}")
            
            if not passed_stage:
                self.passed = False
                print(f"  🛑 Pipeline BLOCKED at stage: {stage.name}")
                if stage.is_quality_gate:
                    print(f"     Quality Gate rejected — investigate metrics before retry")
            
            self.results.append({"stage": stage.name, "status": status, "message": message})
        
        print(f"\n{'='*65}")
        if self.passed:
            print(f"  🎉 PIPELINE PASSED — {self.app} {self.version} deployed to production")
        else:
            failed = [r for r in self.results if r["status"] == "FAIL"]
            print(f"  🛑 PIPELINE FAILED at: {failed[0]['stage']}")
            print(f"     Fix: {failed[0]['message']}")
        print(f"{'='*65}\n")
        return self.passed

# --- Define Quality Gate thresholds ---
QUALITY_GATE = {
    "f1_min": 0.95,
    "faithfulness_min": 0.99,
    "latency_p95_max": 300,
    "task_success_min": 0.90,
}

def check_lint(metrics):
    return True, "flake8: 0 errors, mypy: 0 type errors"

def check_docker_build(metrics):
    return True, "Image built: loanbot:v1.0 (287MB)"

def check_eval_suite(metrics):
    """The critical Quality Gate — checks all eval metrics."""
    issues = []
    if metrics["f1"] < QUALITY_GATE["f1_min"]:
        issues.append(f"F1={metrics['f1']:.1%} < {QUALITY_GATE['f1_min']:.0%}")
    if metrics["faithfulness"] < QUALITY_GATE["faithfulness_min"]:
        issues.append(f"Faithfulness={metrics['faithfulness']:.1%} < {QUALITY_GATE['faithfulness_min']:.0%}")
    if metrics["latency_p95"] > QUALITY_GATE["latency_p95_max"]:
        issues.append(f"P95={metrics['latency_p95']}s > {QUALITY_GATE['latency_p95_max']}s SLA")
    if metrics["task_success"] < QUALITY_GATE["task_success_min"]:
        issues.append(f"TaskSuccess={metrics['task_success']:.1%} < {QUALITY_GATE['task_success_min']:.0%}")
    if issues:
        return False, "REGRESSION DETECTED: " + "; ".join(issues)
    return True, f"All metrics pass: F1={metrics['f1']:.1%}, Faith={metrics['faithfulness']:.1%}, P95={metrics['latency_p95']}s"

def check_staging(metrics):
    return True, "Staging smoke tests passed (5/5)"

def check_canary(metrics):
    return True, "Canary 10% — 1h monitoring: P95=192s, error_rate=0.1% — metrics stable"

def check_production(metrics):
    return True, "Full rollout complete: 100% traffic on v1.0"

# --- Build pipeline ---
pipeline = CICDPipelineSimulator("LoanBot", "v1.0", "main")
pipeline.add_stage(PipelineStage("Code Push & Checkout",    0.5))
pipeline.add_stage(PipelineStage("Lint & Type Check",       3.2, check_lint))
pipeline.add_stage(PipelineStage("Docker Build",            45.0, check_docker_build))
pipeline.add_stage(PipelineStage("Eval Suite",              120.0, check_eval_suite, is_quality_gate=True))
pipeline.add_stage(PipelineStage("Staging Deploy",          15.0, check_staging))
pipeline.add_stage(PipelineStage("Canary Deploy (10%)",     3600.0, check_canary))
pipeline.add_stage(PipelineStage("Production Rollout",      30.0, check_production))

# --- Scenario A: PASS (good metrics) ---
print("\n📋 SCENARIO A: LoanBot v1.0 — All metrics PASS")
good_metrics = {"f1": 0.955, "faithfulness": 0.992, "latency_p95": 187, "task_success": 0.97}
pipeline_a = CICDPipelineSimulator("LoanBot", "v1.0", "main")
for stage in pipeline.stages:
    pipeline_a.add_stage(stage)
pipeline_a.run(good_metrics)

In [ ]:
# --- Scenario B: FAIL at Quality Gate ---
print("\n📋 SCENARIO B: LoanBot v1.1 — Faithfulness REGRESSION")
bad_metrics = {"f1": 0.953, "faithfulness": 0.921, "latency_p95": 195, "task_success": 0.95}
pipeline_b = CICDPipelineSimulator("LoanBot", "v1.1", "feature/new-prompt")
for stage in pipeline.stages:
    pipeline_b.add_stage(stage)
pipeline_b.run(bad_metrics)

print("\n💡 Lesson: Quality Gate blocked deploy của v1.1 vì Faithfulness=92.1% < 99%.")
print("   Root cause: system prompt thay đổi khiến LoanBot bịa credit score.")
print("   Hành động: revert prompt change, investigate, retry pipeline.")

---
## Part 4: Production Monitoring Dashboard

Simulate monitoring data cho LoanBot v1.0 trong 24 giờ production và visualize:
- Request volume theo giờ
- P95 Latency trend
- Faithfulness score
- Alert events với P1/P2/P3 tiers

In [ ]:
import random
import math

try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    MATPLOTLIB_AVAILABLE = True
except ImportError:
    MATPLOTLIB_AVAILABLE = False
    print("⚠️  matplotlib not installed. Run: pip install matplotlib")
    print("   Continuing with text-based dashboard...\n")

random.seed(42)

def simulate_24h_metrics():
    """Simulate LoanBot production metrics over 24 hours."""
    hours = list(range(24))
    # Business hours (8-20): high load; off-hours: low
    def load_factor(h): return 1.0 if 8 <= h <= 20 else 0.1
    
    requests_per_hour = [int(45 * load_factor(h) + random.gauss(0, 3)) for h in hours]
    requests_per_hour = [max(0, r) for r in requests_per_hour]
    
    # P95 latency: normally ~180s, spike at hour 14 (CIC API slow)
    p95_latency = []
    for h in hours:
        base = 180 + random.gauss(0, 10)
        if h == 14:  # CIC API slowdown
            base = 340  # SLA BREACH → P2 Alert
        if h == 9:   # Morning peak
            base = 265
        p95_latency.append(max(50, int(base)))
    
    # Faithfulness: normally >99%, dip at hour 17 (edge case cluster)
    faithfulness = []
    for h in hours:
        val = 0.992 + random.gauss(0, 0.003)
        if h == 17:  # Complex batch triggers edge cases
            val = 0.971  # NEAR threshold → WARNING
        faithfulness.append(round(min(1.0, max(0.9, val)), 4))
    
    # Error rate (0-1%)
    error_rate = [round(max(0, random.gauss(0.003, 0.001)), 4) for _ in hours]
    if 14 in hours:
        error_rate[14] = 0.025  # spike during CIC issue
    
    return {
        "hours": hours,
        "requests": requests_per_hour,
        "p95_latency": p95_latency,
        "faithfulness": faithfulness,
        "error_rate": error_rate
    }

metrics_24h = simulate_24h_metrics()

# Alert detection
def detect_alerts(metrics):
    alerts = []
    for h in metrics["hours"]:
        if metrics["p95_latency"][h] > 300:
            alerts.append({"hour": h, "tier": "P2", "type": "Latency SLA Breach",
                          "value": f"P95={metrics['p95_latency'][h]}s"})
        if metrics["faithfulness"][h] < 0.99:
            tier = "P1" if metrics["faithfulness"][h] < 0.95 else "P3"
            alerts.append({"hour": h, "tier": tier, "type": "Faithfulness Drop",
                          "value": f"{metrics['faithfulness'][h]:.1%}"})
        if metrics["error_rate"][h] > 0.01:
            alerts.append({"hour": h, "tier": "P2", "type": "Error Rate Spike",
                          "value": f"{metrics['error_rate'][h]:.1%}"})
    return alerts

alerts = detect_alerts(metrics_24h)

# Text dashboard (always shown)
print("\n" + "="*65)
print("  LoanBot v1.0 — 24h Production Dashboard (2026-06-26)")
print("="*65)
total_req = sum(metrics_24h["requests"])
avg_p95   = sum(metrics_24h["p95_latency"]) / 24
avg_faith = sum(metrics_24h["faithfulness"]) / 24
print(f"  Total requests  : {total_req:,} hồ sơ")
print(f"  Avg P95 latency : {avg_p95:.0f}s (SLA: ≤300s)")
print(f"  Avg Faithfulness: {avg_faith:.2%} (SLA: ≥99%)")
print(f"  Alerts fired    : {len(alerts)}")

print(f"\n  {'Hour':>5} {'Requests':>10} {'P95(s)':>8} {'Faith%':>8} {'ErrRate':>8}")
print(f"  {'─'*50}")
for h in metrics_24h["hours"]:
    flag = ""
    if metrics_24h["p95_latency"][h] > 300: flag += " ⚠️P2"
    if metrics_24h["faithfulness"][h] < 0.99: flag += " ⚠️P3"
    print(f"  {h:>4}h {metrics_24h['requests'][h]:>10} {metrics_24h['p95_latency'][h]:>8}s "
          f"{metrics_24h['faithfulness'][h]:>7.1%} {metrics_24h['error_rate'][h]:>7.1%}{flag}")

print(f"\n  ALERTS:")
for a in alerts:
    tier_icon = {"P1": "🔴", "P2": "🟠", "P3": "🟡"}.get(a["tier"], "⚪")
    print(f"  {tier_icon} [{a['tier']}] Hour {a['hour']:02d}:00 — {a['type']}: {a['value']}")

In [ ]:
if MATPLOTLIB_AVAILABLE:
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    fig.suptitle("LoanBot v1.0 — Production Dashboard (24h)", fontsize=14, fontweight='bold')
    hours = metrics_24h["hours"]
    
    # Plot 1: Request Volume
    ax1 = axes[0, 0]
    ax1.bar(hours, metrics_24h["requests"], color='#0f4c81', alpha=0.7)
    ax1.set_title("Request Volume (hồ sơ/giờ)")
    ax1.set_xlabel("Hour"); ax1.set_ylabel("Requests")
    ax1.axvspan(8, 20, alpha=0.05, color='orange', label='Business hours')
    ax1.legend(fontsize=8)
    
    # Plot 2: P95 Latency
    ax2 = axes[0, 1]
    colors = ['red' if v > 300 else '#0f4c81' for v in metrics_24h["p95_latency"]]
    ax2.bar(hours, metrics_24h["p95_latency"], color=colors, alpha=0.7)
    ax2.axhline(y=300, color='red', linestyle='--', linewidth=1.5, label='SLA (300s)')
    ax2.set_title("P95 Latency (s)")
    ax2.set_xlabel("Hour"); ax2.set_ylabel("Seconds")
    ax2.legend(fontsize=8)
    for h in hours:
        if metrics_24h["p95_latency"][h] > 300:
            ax2.annotate("⚠️P2", (h, metrics_24h["p95_latency"][h]+5), fontsize=8, ha='center', color='red')
    
    # Plot 3: Faithfulness
    faith_pct = [f * 100 for f in metrics_24h["faithfulness"]]
    ax3 = axes[1, 0]
    ax3.plot(hours, faith_pct, color='#16a34a', linewidth=2, marker='o', markersize=4)
    ax3.axhline(y=99, color='red', linestyle='--', linewidth=1.5, label='SLA (99%)')
    ax3.fill_between(hours, faith_pct, 99, where=[f < 99 for f in faith_pct],
                     alpha=0.3, color='orange', label='Below SLA')
    ax3.set_ylim(96, 101)
    ax3.set_title("Faithfulness Score (%)")
    ax3.set_xlabel("Hour"); ax3.set_ylabel("Faithfulness %")
    ax3.legend(fontsize=8)
    
    # Plot 4: Alert Timeline
    ax4 = axes[1, 1]
    tier_colors = {"P1": "red", "P2": "orange", "P3": "gold"}
    for a in alerts:
        ax4.scatter(a["hour"], {"P1": 3, "P2": 2, "P3": 1}.get(a["tier"], 0),
                   color=tier_colors.get(a["tier"], "gray"), s=200, zorder=5)
        ax4.annotate(a["type"][:15], (a["hour"], {"P1": 3, "P2": 2, "P3": 1}.get(a["tier"], 0)),
                    textcoords="offset points", xytext=(0, 10), fontsize=7, ha='center')
    ax4.set_xlim(-1, 24); ax4.set_ylim(0, 4)
    ax4.set_yticks([1, 2, 3]); ax4.set_yticklabels(["P3", "P2", "P1"])
    ax4.set_title("Alert Timeline")
    ax4.set_xlabel("Hour")
    legend_patches = [mpatches.Patch(color=v, label=k) for k, v in tier_colors.items()]
    ax4.legend(handles=legend_patches, fontsize=8)
    
    plt.tight_layout()
    plt.savefig("loanbot_dashboard.png", dpi=100, bbox_inches='tight')
    plt.show()
    print("\n✅ Dashboard saved to loanbot_dashboard.png")
else:
    print("Install matplotlib to see the visual dashboard: pip install matplotlib")

---
## Part 5: Scaling & Cost Calculator

FinTech Corp cần quyết định:
1. Bao nhiêu replicas cho business hours vs off-hours?
2. Dùng model tier nào (Haiku vs Sonnet)?
3. ROI so với hệ thống cũ (200 nhân viên)?

In [ ]:
from dataclasses import dataclass
from typing import Dict, Tuple

@dataclass
class ReplicaConfig:
    min_replicas: int
    max_replicas: int
    cpu_per_replica: float  # cores
    ram_per_replica: float  # GB
    cost_per_replica_per_hour: float  # USD

@dataclass
class ModelTier:
    name: str
    input_cost_per_m: float   # USD per 1M input tokens
    output_cost_per_m: float  # USD per 1M output tokens
    avg_tokens_per_request: int  # input + output
    suitable_for: str

class ScalingCostCalculator:
    BUSINESS_HOURS_PER_DAY = 14   # 8AM - 10PM
    OFF_HOURS_PER_DAY = 10
    WORKING_DAYS_PER_MONTH = 22
    
    def __init__(self, replica_config: ReplicaConfig, peak_requests_per_hour: int):
        self.rc = replica_config
        self.peak_rph = peak_requests_per_hour
        
    def requests_per_replica_capacity(self) -> float:
        # Assume each replica handles 1 request concurrently, avg 180s → ~20 req/hour
        return 3600 / 187  # = ~19.3 requests/hour per replica
    
    def required_replicas(self, requests_per_hour: int) -> int:
        capacity = self.requests_per_replica_capacity()
        needed = math.ceil(requests_per_hour / capacity)
        return max(self.rc.min_replicas, min(self.rc.max_replicas, needed))
    
    def monthly_infra_cost(self) -> Dict[str, float]:
        peak_replicas = self.required_replicas(self.peak_rph)
        off_replicas = self.rc.min_replicas
        
        hours_peak = self.BUSINESS_HOURS_PER_DAY * self.WORKING_DAYS_PER_MONTH
        hours_off  = self.OFF_HOURS_PER_DAY * self.WORKING_DAYS_PER_MONTH
        # Add weekends (all off-hours)
        weekend_days = 30 - self.WORKING_DAYS_PER_MONTH
        hours_off += weekend_days * 24
        
        cost_peak = peak_replicas * self.rc.cost_per_replica_per_hour * hours_peak
        cost_off  = off_replicas  * self.rc.cost_per_replica_per_hour * hours_off
        
        return {
            "peak_replicas": peak_replicas,
            "off_replicas": off_replicas,
            "hours_peak": hours_peak,
            "hours_off": hours_off,
            "cost_peak_usd": cost_peak,
            "cost_off_usd": cost_off,
            "total_infra_usd": cost_peak + cost_off,
        }
    
    def monthly_api_cost(self, total_requests: int, model: ModelTier) -> Dict[str, float]:
        # Estimate token split: 70% input, 30% output
        input_tokens  = model.avg_tokens_per_request * 0.7 * total_requests
        output_tokens = model.avg_tokens_per_request * 0.3 * total_requests
        input_cost  = (input_tokens  / 1_000_000) * model.input_cost_per_m
        output_cost = (output_tokens / 1_000_000) * model.output_cost_per_m
        return {
            "input_tokens_m": input_tokens / 1_000_000,
            "output_tokens_m": output_tokens / 1_000_000,
            "input_cost_usd": input_cost,
            "output_cost_usd": output_cost,
            "total_api_usd": input_cost + output_cost,
        }
    
    def roi_vs_manual(self, monthly_requests: int, manual_cost_per_case_vnd: int,
                      usd_to_vnd: int = 25500) -> Dict:
        manual_total_vnd = monthly_requests * manual_cost_per_case_vnd
        infra = self.monthly_infra_cost()
        haiku = ModelTier("claude-haiku-4-5", 0.25, 1.25, 8000, "simple cases")
        api = self.monthly_api_cost(monthly_requests, haiku)
        ai_total_vnd = (infra["total_infra_usd"] + api["total_api_usd"]) * usd_to_vnd
        savings_vnd = manual_total_vnd - ai_total_vnd
        roi_pct = savings_vnd / manual_total_vnd * 100
        return {
            "manual_total_vnd": manual_total_vnd,
            "ai_total_vnd": ai_total_vnd,
            "savings_vnd": savings_vnd,
            "roi_pct": roi_pct,
            "infra_usd": infra["total_infra_usd"],
            "api_usd": api["total_api_usd"],
        }

# --- FinTech Corp configuration ---
import math

rc = ReplicaConfig(
    min_replicas=2,
    max_replicas=8,
    cpu_per_replica=2.0,
    ram_per_replica=4.0,
    cost_per_replica_per_hour=0.12  # USD/hour (e.g., AWS t3.medium equivalent)
)

calc = ScalingCostCalculator(rc, peak_requests_per_hour=50)

models = [
    ModelTier("claude-haiku-4-5-20251001",  0.25,  1.25, 8_000, "80% simple cases (score≥700)"),
    ModelTier("claude-sonnet-4-6",          3.00, 15.00, 10_000, "20% complex cases"),
]

MONTHLY_REQUESTS = 200_000
MANUAL_COST_PER_CASE_VND = 350_000  # 350K VNĐ/hồ sơ

print("\n" + "="*65)
print("  FinTech Corp LoanBot v1.0 — Scaling & Cost Analysis")
print("="*65)

# Infra
infra = calc.monthly_infra_cost()
print(f"\n📊 REPLICA SCALING:")
print(f"  Peak hours (8AM-10PM): {infra['peak_replicas']} replicas × {infra['hours_peak']} hours/month")
print(f"  Off-peak:              {infra['off_replicas']} replicas × {infra['hours_off']} hours/month")
print(f"  Infra cost (peak):     ${infra['cost_peak_usd']:,.0f}/month")
print(f"  Infra cost (off-peak): ${infra['cost_off_usd']:,.0f}/month")
print(f"  TOTAL INFRA:           ${infra['total_infra_usd']:,.0f}/month")

# API costs per model
print(f"\n💰 API COST ({MONTHLY_REQUESTS:,} requests/month):")
for m in models:
    api = calc.monthly_api_cost(MONTHLY_REQUESTS, m)
    print(f"  {m.name}")
    print(f"    → Suitable for: {m.suitable_for}")
    print(f"    → Input tokens: {api['input_tokens_m']:.1f}M @ ${m.input_cost_per_m}/M = ${api['input_cost_usd']:,.0f}")
    print(f"    → Output tokens: {api['output_tokens_m']:.1f}M @ ${m.output_cost_per_m}/M = ${api['output_cost_usd']:,.0f}")
    print(f"    → TOTAL API: ${api['total_api_usd']:,.0f}/month")

# Hybrid tiering
haiku_api = calc.monthly_api_cost(int(MONTHLY_REQUESTS * 0.8), models[0])
sonnet_api = calc.monthly_api_cost(int(MONTHLY_REQUESTS * 0.2), models[1])
hybrid_api = haiku_api["total_api_usd"] + sonnet_api["total_api_usd"]
print(f"\n  🎯 HYBRID TIERING (80% Haiku + 20% Sonnet):")
print(f"    → Haiku (80%): ${haiku_api['total_api_usd']:,.0f}")
print(f"    → Sonnet (20%): ${sonnet_api['total_api_usd']:,.0f}")
print(f"    → TOTAL: ${hybrid_api:,.0f}/month (vs ${calc.monthly_api_cost(MONTHLY_REQUESTS, models[1])['total_api_usd']:,.0f} all-Sonnet)")

# ROI
roi = calc.roi_vs_manual(MONTHLY_REQUESTS, MANUAL_COST_PER_CASE_VND)
print(f"\n🏆 ROI vs MANUAL PROCESSING:")
print(f"  Manual (200 nhân viên × 350K/hồ sơ):  {roi['manual_total_vnd']/1e9:.1f} tỷ VNĐ/tháng")
print(f"  LoanBot AI (infra + API):              {roi['ai_total_vnd']/1e6:.0f} triệu VNĐ/tháng")
print(f"  Monthly savings:                       {roi['savings_vnd']/1e9:.2f} tỷ VNĐ")
print(f"  ROI:                                   {roi['roi_pct']:.1f}%")
print(f"  ROI (yearly):                          {roi['savings_vnd']*12/1e9:.0f} tỷ VNĐ")
print()

---
## Part 6: Real Claude API — Production Config Test

Tích hợp thực tế với Claude API, test LoanBot trong production-like config với observability:

In [ ]:
import os
import json
import time
import uuid
from dataclasses import dataclass, field, asdict
from typing import Any, Dict, List, Optional
from enum import Enum

try:
    import anthropic
    ANTHROPIC_AVAILABLE = True
except ImportError:
    ANTHROPIC_AVAILABLE = False
    print("⚠️  anthropic not installed. Run: pip install anthropic")

# ── Production-like tool definitions ──────────────────────────────
TOOLS = [
    {"name": "check_credit_score",
     "description": "Truy vấn điểm tín dụng từ CIC (Credit Information Center VN)",
     "input_schema": {"type": "object",
                      "properties": {"customer_id": {"type": "string"},
                                     "national_id": {"type": "string"}},
                      "required": ["customer_id", "national_id"]}},
    {"name": "verify_income",
     "description": "Xác minh thu nhập qua ngân hàng và thuế",
     "input_schema": {"type": "object",
                      "properties": {"customer_id": {"type": "string"},
                                     "declared_monthly_income": {"type": "number"}},
                      "required": ["customer_id", "declared_monthly_income"]}},
    {"name": "calculate_dti",
     "description": "Tính Debt-to-Income ratio",
     "input_schema": {"type": "object",
                      "properties": {"monthly_income": {"type": "number"},
                                     "existing_monthly_debt": {"type": "number"},
                                     "new_loan_monthly_payment": {"type": "number"}},
                      "required": ["monthly_income", "existing_monthly_debt", "new_loan_monthly_payment"]}},
    {"name": "check_blacklist",
     "description": "Kiểm tra danh sách đen tín dụng quốc gia",
     "input_schema": {"type": "object",
                      "properties": {"customer_id": {"type": "string"}},
                      "required": ["customer_id"]}},
    {"name": "generate_report",
     "description": "Tạo báo cáo phê duyệt hoặc từ chối hồ sơ vay",
     "input_schema": {"type": "object",
                      "properties": {"customer_id": {"type": "string"},
                                     "decision": {"type": "string", "enum": ["APPROVED", "REJECTED", "PENDING_REVIEW"]},
                                     "loan_amount_vnd": {"type": "number"},
                                     "credit_score": {"type": "number"},
                                     "dti_ratio": {"type": "number"},
                                     "reason": {"type": "string"}},
                      "required": ["customer_id", "decision", "reason"]}},
]

# ── Tool implementations (mock CIC, income, etc.) ─────────────────
MOCK_DB = {
    "FC-2024-001": {"credit_score": 720, "income_verified": 35_000_000, "blacklisted": False},
    "FC-2024-002": {"credit_score": 580, "income_verified": 12_000_000, "blacklisted": False},
    "FC-2024-003": {"credit_score": 400, "income_verified": 8_000_000,  "blacklisted": True},
}

def execute_tool(name: str, inputs: dict) -> dict:
    if name == "check_credit_score":
        cid = inputs["customer_id"]
        data = MOCK_DB.get(cid, {"credit_score": 0})
        return {"customer_id": cid, "credit_score": data["credit_score"],
                "grade": "A" if data["credit_score"] >= 700 else "B" if data["credit_score"] >= 600 else "C"}
    elif name == "verify_income":
        cid = inputs["customer_id"]
        data = MOCK_DB.get(cid, {})
        verified = data.get("income_verified", 0)
        return {"customer_id": cid, "declared": inputs["declared_monthly_income"],
                "verified": verified, "discrepancy_pct": abs(verified - inputs["declared_monthly_income"]) / max(verified, 1) * 100}
    elif name == "calculate_dti":
        total_debt = inputs["existing_monthly_debt"] + inputs["new_loan_monthly_payment"]
        dti = total_debt / inputs["monthly_income"]
        return {"dti_ratio": round(dti, 3), "total_monthly_debt": total_debt,
                "is_acceptable": dti <= 0.43}
    elif name == "check_blacklist":
        cid = inputs["customer_id"]
        data = MOCK_DB.get(cid, {"blacklisted": False})
        return {"customer_id": cid, "blacklisted": data["blacklisted"],
                "checked_at": "2026-06-26T10:00:00"}
    elif name == "generate_report":
        return {"report_id": f"RPT-{inputs['customer_id']}-{uuid.uuid4().hex[:6].upper()}",
                "decision": inputs["decision"], "customer_id": inputs["customer_id"],
                "loan_amount_vnd": inputs.get("loan_amount_vnd", 0),
                "reason": inputs["reason"], "generated_at": "2026-06-26T10:00:00"}
    return {"error": f"Unknown tool: {name}"}

# ── Production LoanBot with observability ─────────────────────────
class ProductionLoanBot:
    SYSTEM_PROMPT = """Bạn là LoanBot v1.0 của FinTech Corp — AI agent phân tích hồ sơ vay vốn.

QUY TRÌNH BẮTBUỘC (theo thứ tự):
1. check_blacklist → nếu blacklisted, REJECTED ngay
2. check_credit_score → credit grade
3. verify_income → xác minh thu nhập
4. calculate_dti → tính DTI ratio
5. generate_report → quyết định cuối

TIÊU CHUẨN PHÊ DUYỆT:
- Credit score ≥ 650 và DTI ≤ 43%: APPROVED
- Credit score 580-649 hoặc DTI 43-50%: PENDING_REVIEW
- Credit score < 580 hoặc DTI > 50% hoặc blacklisted: REJECTED

QUAN TRỌNG: Chỉ sử dụng con số từ tool results. Không bịa số liệu."""
    
    def __init__(self):
        self.tool_calls_log = []
    
    def run(self, loan_request: str, trace_id: str = None) -> dict:
        if not ANTHROPIC_AVAILABLE:
            return {"error": "anthropic not installed"}
        
        api_key = os.environ.get("ANTHROPIC_API_KEY")
        if not api_key:
            return {"error": "ANTHROPIC_API_KEY not set"}
        
        client = anthropic.Anthropic(api_key=api_key)
        trace_id = trace_id or f"TRC-{uuid.uuid4().hex[:8].upper()}"
        messages = [{"role": "user", "content": loan_request}]
        start_time = time.time()
        total_input_tokens = 0
        total_output_tokens = 0
        iterations = 0
        
        print(f"\n🤖 LoanBot v1.0 | trace_id={trace_id}")
        print(f"{'─'*55}")
        
        while iterations < 10:
            iterations += 1
            response = client.messages.create(
                model="claude-haiku-4-5-20251001",
                max_tokens=1024,
                system=self.SYSTEM_PROMPT,
                tools=TOOLS,
                messages=messages
            )
            total_input_tokens  += response.usage.input_tokens
            total_output_tokens += response.usage.output_tokens
            
            if response.stop_reason == "end_turn":
                final_text = ""
                for block in response.content:
                    if hasattr(block, "text"):
                        final_text = block.text
                latency = time.time() - start_time
                cost_usd = (total_input_tokens / 1e6 * 0.25 + total_output_tokens / 1e6 * 1.25)
                print(f"\n📄 FINAL DECISION:")
                print(final_text[:600])
                print(f"\n📊 METRICS:")
                print(f"  Latency    : {latency:.1f}s")
                print(f"  Iterations : {iterations}")
                print(f"  Tool calls : {len(self.tool_calls_log)}")
                print(f"  Tokens     : {total_input_tokens} in / {total_output_tokens} out")
                print(f"  Cost       : ${cost_usd:.5f} USD")
                return {"trace_id": trace_id, "final_text": final_text,
                        "latency_s": latency, "iterations": iterations,
                        "tool_calls": len(self.tool_calls_log), "cost_usd": cost_usd}
            
            # Process tool calls
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    t_start = time.time()
                    result = execute_tool(block.name, block.input)
                    t_ms = (time.time() - t_start) * 1000
                    print(f"  🔧 {block.name}({json.dumps(block.input)[:60]}) → {json.dumps(result)[:60]}")
                    self.tool_calls_log.append({"tool": block.name, "latency_ms": round(t_ms, 1)})
                    tool_results.append({"type": "tool_result", "tool_use_id": block.id,
                                         "content": json.dumps(result)})
            
            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": tool_results})
        
        return {"error": "Max iterations reached"}

# Run test
bot = ProductionLoanBot()
request = """
Phân tích hồ sơ vay vốn:
- Customer ID: FC-2024-001 (Nguyễn Văn An)
- CCCD: 001234567890  
- Thu nhập khai báo: 35,000,000 VNĐ/tháng
- Nợ hiện tại: 5,000,000 VNĐ/tháng
- Khoản vay: 300,000,000 VNĐ, 36 tháng, lãi suất 10%/năm
"""
result = bot.run(request)
if "error" in result:
    print(f"Note: {result['error']}")
    print("Set ANTHROPIC_API_KEY environment variable to run with real API.")

---
## 💡 Exercises

### Exercise 1: Failing Checklist
Modify the Production Readiness Check to simulate LoanBot v0.9 where:
- AI Disclosure status = FAIL (compliance team never signed off)
- Faithfulness Score = 97% (below 99% threshold)

What does the verdict show? Which items are blocking deployment?

In [ ]:
# Exercise 1: Simulate v0.9 with compliance failures
checker_v09 = ProductionReadinessChecker("LoanBot", "v0.9")

# TODO: record checks for v0.9 — set AI Disclosure = FAIL, Faithfulness = WARNING
# Hint: use checker_v09.record_check(name, CheckStatus.FAIL, actual, note)
# All other checks can be PASS (copy from the v1.0 example above)

# checker_v09.record_check("AI Disclosure", ???, ???)
# checker_v09.record_check("Faithfulness Score", ???, "Faithfulness = 97.0%")
# ... fill in the rest ...

# checker_v09.print_report()

print("TODO: Complete Exercise 1")

### Exercise 2: Security Scanner
Extend the `DockerfileGenerator.security_scan()` to also detect:
- Running as root (`USER root` or missing USER directive)
- Outdated base image Python version < 3.10
- `--no-verify-ssl` or `--trusted-host` in pip install (insecure)

Test with a deliberately insecure config.

In [ ]:
# Exercise 2: Enhanced security scanner
class EnhancedDockerfileGenerator(DockerfileGenerator):
    def security_scan(self) -> List[str]:
        issues = super().security_scan()  # keep parent checks
        dockerfile = self.generate_dockerfile()
        
        # TODO: Add 3 new security checks:
        # 1. Check python version < 3.10
        # 2. Check for --no-verify-ssl or --trusted-host in pip install
        # 3. Check for missing USER directive (running as root)
        
        return issues

# Test with insecure config
insecure_docker = EnhancedDockerfileGenerator(DockerConfig(
    app_name="loanbot-insecure",
    python_version="3.8",  # Old version
    port=8080,
    health_check_path="/health",
    env_secrets=[],
    env_config={"ANTHROPIC_API_KEY": "sk-ant-secret123"},  # BAD! hardcoded secret
    requirements=["anthropic"],
    main_module="loanbot.api"
))

# TODO: print security scan results
print("TODO: Complete Exercise 2")

### Exercise 3: Alert Tuning
The current monitoring generates too many P3 alerts (every time faithfulness drops below 99% even by 0.1%). 

Design a better alert strategy:
- P3 only if 3 consecutive hours below 99%
- P2 if any single hour below 97%
- P1 if below 95%

Rerun the 24h simulation with the new strategy. How many fewer alerts?

In [ ]:
# Exercise 3: Smarter alert logic
def detect_alerts_smart(metrics: dict) -> List[dict]:
    alerts = []
    # TODO: implement consecutive-hour check for P3
    # Hint: use a sliding window of 3 hours
    # - P3: 3 consecutive hours with faithfulness < 0.99
    # - P2: any single hour with faithfulness < 0.97
    # - P1: any single hour with faithfulness < 0.95
    # Keep existing latency and error rate logic
    return alerts

# Compare
original_alerts = detect_alerts(metrics_24h)
smart_alerts    = detect_alerts_smart(metrics_24h)
print(f"Original alerts: {len(original_alerts)}")
print(f"Smart alerts:    {len(smart_alerts)} (TODO)")
print("TODO: Complete Exercise 3")

---
## 🏁 Module Summary

Trong lab này, bạn đã:

| Task | Kỹ năng | Tool/API |
|------|---------|----------|
| Production Readiness Check | 15-item checklist với blocking logic | ProductionReadinessChecker |
| Docker Configuration | Multi-stage, non-root, HEALTHCHECK | DockerfileGenerator |
| CI/CD Pipeline | 7-stage với Quality Gate | CICDPipelineSimulator |
| Monitoring Dashboard | P1/P2/P3 alerts, 24h visualize | matplotlib |
| Scaling Calculator | Replica planning, model tiering, ROI | ScalingCostCalculator |
| Production API Test | Real Claude API với observability | anthropic SDK |

### Key Takeaways

1. **Compliance checks are BLOCKING** — không thể deploy LoanBot nếu Legal chưa sign-off
2. **Quality Gate = nhân vật chính của CI/CD** — F1 drop 3% = stop pipeline
3. **Secrets không bao giờ hardcode** — dùng Secrets Manager, inject at runtime
4. **Model tiering = cost optimization** — 80% Haiku + 20% Sonnet = ~65% cost reduction
5. **ROI = 99.7%** — AI agent thay thế 200 nhân viên với quality cao hơn

### Chuẩn bị cho Module 5
Module 5 sẽ đi sâu vào **Governance & Security**:
- EU AI Act compliance cho LoanBot (high-risk AI system)
- Security attack vectors: prompt injection, data poisoning
- Privacy-preserving AI: differential privacy, data minimization
- Responsible AI audit framework